# AVL Tree (Self-Balancing BST)

A plain [binary search tree](../trees/binary-search-tree.ipynb) gives `O(h)` operations, but `h` can degrade to `O(n)` when keys arrive in sorted order -- the tree becomes a linked list. An **AVL tree** (Adelson-Velsky and Landis, 1962) is a BST that **rebalances itself after every insert and delete** so that `h` stays `O(log n)`.

## The balance invariant

For **every** node, the heights of its left and right subtrees differ by at most 1:

```
balance_factor(node) = height(left subtree) - height(right subtree)
```

A node is *balanced* when its balance factor is `-1`, `0`, or `+1`. If an insert or delete pushes any node's balance factor to `+2` or `-2`, we restore the invariant with **rotations**.

| Operation | Time | Aux space |
|-----------|------|-----------|
| Search | `O(log n)` | `O(log n)` recursion / `O(1)` iterative |
| Insert | `O(log n)` | `O(log n)` |
| Delete | `O(log n)` | `O(log n)` |

Search is identical to a normal BST (the ordering invariant is unchanged), so this notebook focuses on **height tracking, rotations, and rebalancing**.

### Height bookkeeping

Every node **stores** its height rather than computing it on demand. That is what keeps
rebalancing cheap: a balance factor is then two O(1) lookups instead of two subtree walks,
which would make every check O(n).

The cost of caching is that the cache must be maintained -- `update_height` has to be called
after *any* structural change, and always bottom-up, since a parent's height is defined in
terms of its children's.

Conventions used here:

- `height(None) = 0`, a leaf has height 1 (counting nodes, matching the binary tree notebook)
- `balance_factor = height(left) - height(right)`, so **positive means left-heavy**
- `balance_factor(None) = 0` -- a missing subtree is perfectly balanced, which keeps the
  callers free of null checks

**Time:** O(1) for all three helpers

In [ ]:
class Node:
    def __init__(self, data):
        self.data = data
        self.left = None
        self.right = None
        self.height = 1  # height of a new leaf is 1

def height(node):
    """Height of None is 0; height is stored on the node and kept up to date."""
    return node.height if node else 0

def update_height(node):
    """Recompute a node's height from its children. Call after any structural change."""
    node.height = 1 + max(height(node.left), height(node.right))

def balance_factor(node):
    """left height - right height. Positive => left heavy, negative => right heavy."""
    if node is None:
        return 0
    return height(node.left) - height(node.right)

# Rotations

A rotation is a local, `O(1)` rearrangement of a few pointers that changes the shape of the tree **without breaking the BST ordering**. There are two primitives -- left and right -- and they are mirror images of each other.

### Right rotation (fixes a left-heavy node)

```
      y                x
     / \              / \
    x   T3   --->    T1  y
   / \                  / \
  T1  T2               T2  T3
```

`x` moves up, `y` becomes its right child, and `x`'s old right subtree `T2` is reattached as `y`'s left child. Ordering is preserved: `T1 < x < T2 < y < T3` before and after.

### Left rotation (fixes a right-heavy node)

```
    x                  y
   / \                / \
  T1  y     --->     x   T3
     / \            / \
    T2  T3         T1  T2
```

The exact mirror. After either rotation we recompute the heights of the two nodes that moved, **bottom-up** (the lower node first).

In [ ]:
def rotate_right(y):
    """
    Right-rotate around y. Returns the new subtree root (x).
    Time: O(1). Used to fix a left-heavy node.
    """
    x = y.left
    t2 = x.right
    # rotate
    x.right = y
    y.left = t2
    # update heights bottom-up: y first (now lower), then x
    update_height(y)
    update_height(x)
    return x

def rotate_left(x):
    """
    Left-rotate around x. Returns the new subtree root (y).
    Time: O(1). Used to fix a right-heavy node.
    """
    y = x.right
    t2 = y.left
    # rotate
    y.left = x
    x.right = t2
    # update heights bottom-up: x first (now lower), then y
    update_height(x)
    update_height(y)
    return y

# The four imbalance cases

When a node becomes unbalanced (`|balance_factor| > 1`), exactly one of four cases applies. They are named by the path from the unbalanced node to the newly-inserted node.

| Case | Condition | Fix |
|------|-----------|-----|
| **Left-Left (LL)** | node is left-heavy, left child is left-heavy/balanced | `rotate_right(node)` |
| **Left-Right (LR)** | node is left-heavy, left child is right-heavy | `rotate_left(left)` then `rotate_right(node)` |
| **Right-Right (RR)** | node is right-heavy, right child is right-heavy/balanced | `rotate_left(node)` |
| **Right-Left (RL)** | node is right-heavy, right child is left-heavy | `rotate_right(right)` then `rotate_left(node)` |

The LR and RL cases are "double rotations": a first rotation on the child reduces them to the LL/RR case, which the second rotation then fixes.

```
Left-Right (LR):  rotate_left on the left child turns it into Left-Left

      z                z                 y
     /                /                 / \
    x       -->      y        -->      x   z
     \              /
      y            x
```

In [ ]:
def rebalance(node):
    """
    Restore the AVL invariant at `node` after a child changed.
    Updates height, then applies the LL / LR / RR / RL fix if needed.
    Returns the (possibly new) subtree root. Time: O(1).
    """
    update_height(node)
    bf = balance_factor(node)

    # Left heavy (bf == +2)
    if bf > 1:
        if balance_factor(node.left) < 0:     # Left-Right: reduce to Left-Left
            node.left = rotate_left(node.left)
        return rotate_right(node)             # Left-Left

    # Right heavy (bf == -2)
    if bf < -1:
        if balance_factor(node.right) > 0:    # Right-Left: reduce to Right-Right
            node.right = rotate_right(node.right)
        return rotate_left(node)              # Right-Right

    return node  # already balanced

# Insert

Identical to BST insertion, with one addition: as the recursion **unwinds**, every ancestor
of the inserted node calls `rebalance`.

That ordering is the whole trick. The recursive call returns before `rebalance(root)` runs,
so nodes are checked bottom-up -- heights below are already correct by the time a node looks
at its own balance factor. A single insert can unbalance several ancestors, but fixing the
lowest one restores the subtree's original height, which leaves everything above it balanced
too. One rotation (single or double) per insert is always enough.

```
insert 10, 20, 30 into an empty tree

after 10        10                    balanced
after 20        10                    bf(10) = 0 - 1 = -1, still fine
                  \
                   20
after 30        10                    bf(10) = 0 - 2 = -2 → right heavy,
                  \                   right child is right heavy → RR case
                   20
                     \                rotate_left(10):
                      30
                                          20
                                         /  \
                                       10    30      height 2, balanced
```

`rebalance` returns the new subtree root, which is why the caller must assign it back:
`root.left = insert(root.left, data)`. Dropping that assignment silently discards rotations.

**Time:** O(log n) &nbsp; **Space:** O(log n) recursion stack

In [ ]:
def insert(root, data):
    """
    Insert into the AVL tree, rebalancing on the way up.
    Time: O(log n). Aux space: O(log n) recursion stack.
    """
    # 1. standard BST insert
    if root is None:
        return Node(data)
    if data < root.data:
        root.left = insert(root.left, data)
    elif data > root.data:
        root.right = insert(root.right, data)
    else:
        return root  # duplicates not allowed

    # 2. rebalance this ancestor
    return rebalance(root)

def inorder(root, acc):
    if root:
        inorder(root.left, acc)
        acc.append(root.data)
        inorder(root.right, acc)

In [ ]:
def test_rotation_ll():
    # 30, 20, 10 arrive sorted-descending -> would skew left.
    # A single right rotation at the root fixes it.
    root = None
    for v in [30, 20, 10]:
        root = insert(root, v)
    assert root.data == 20
    assert root.left.data == 10
    assert root.right.data == 30
    assert root.height == 2

test_rotation_ll()

In [ ]:
def test_rotation_rr():
    # 10, 20, 30 ascending -> right-skewed -> single left rotation.
    root = None
    for v in [10, 20, 30]:
        root = insert(root, v)
    assert root.data == 20
    assert root.left.data == 10
    assert root.right.data == 30

test_rotation_rr()

In [ ]:
def test_rotation_lr():
    # 30, 10, 20 -> Left-Right: rotate_left(left) then rotate_right(root).
    root = None
    for v in [30, 10, 20]:
        root = insert(root, v)
    assert root.data == 20
    assert root.left.data == 10
    assert root.right.data == 30

test_rotation_lr()

In [ ]:
def test_rotation_rl():
    # 10, 30, 20 -> Right-Left: rotate_right(right) then rotate_left(root).
    root = None
    for v in [10, 30, 20]:
        root = insert(root, v)
    assert root.data == 20
    assert root.left.data == 10
    assert root.right.data == 30

test_rotation_rl()

# Checking the invariant

A small helper that recursively verifies every node satisfies `|balance_factor| <= 1`. We use it in the tests below to assert the tree stays balanced no matter the insertion order.

In [ ]:
def is_avl_balanced(node):
    """True if every node in the subtree has balance factor in {-1, 0, 1}."""
    if node is None:
        return True
    if abs(balance_factor(node)) > 1:
        return False
    return is_avl_balanced(node.left) and is_avl_balanced(node.right)

def test_stays_balanced():
    root = None
    for v in [10, 20, 30, 40, 50, 25]:
        root = insert(root, v)
    res = []
    inorder(root, res)
    assert res == [10, 20, 25, 30, 40, 50]  # still a valid BST
    assert is_avl_balanced(root)
    assert root.height == 3  # 6 nodes balanced; a plain BST here would be height 5

test_stays_balanced()

In [ ]:
def test_sequential_inserts_stay_log_height():
    # The pathological case for a plain BST: 1..63 in ascending order
    # would build a height-63 linked list. AVL keeps it logarithmic.
    root = None
    for v in range(1, 64):
        root = insert(root, v)
    assert is_avl_balanced(root)
    assert root.height == 6  # 63 nodes -> perfectly balanced height
    res = []
    inorder(root, res)
    assert res == list(range(1, 64))

test_sequential_inserts_stay_log_height()

# Delete

Like BST delete (three cases: leaf, one child, two children -- replacing with the inorder successor), but every ancestor calls `rebalance` as the recursion unwinds. A deletion can require rebalancing at multiple levels, but each fix is still `O(1)` and the total stays `O(log n)`.

In [ ]:
def min_node(node):
    """Leftmost (smallest) node in a subtree."""
    while node.left is not None:
        node = node.left
    return node

def delete(root, data):
    """
    Delete from the AVL tree, rebalancing on the way up.
    Time: O(log n). Aux space: O(log n).
    """
    if root is None:
        return None

    # 1. standard BST delete
    if data < root.data:
        root.left = delete(root.left, data)
    elif data > root.data:
        root.right = delete(root.right, data)
    else:
        if root.left is None:
            return root.right
        if root.right is None:
            return root.left
        # two children: replace with inorder successor, then delete it
        succ = min_node(root.right)
        root.data = succ.data
        root.right = delete(root.right, succ.data)

    # 2. rebalance this ancestor
    return rebalance(root)

def test_delete_rebalances():
    root = None
    for v in [10, 20, 30, 40, 50, 25]:
        root = insert(root, v)
    root = delete(root, 10)
    res = []
    inorder(root, res)
    assert res == [20, 25, 30, 40, 50]
    assert is_avl_balanced(root)

    root = delete(root, 40)
    res = []
    inorder(root, res)
    assert res == [20, 25, 30, 50]
    assert is_avl_balanced(root)

test_delete_rebalances()

# Python Built-in Note

Python has **no built-in balanced BST**. The standard-library `bisect` module keeps a plain list sorted with `O(log n)` *search* but `O(n)` *insert/delete* (array shifting) -- see the [binary-search](../searching/binary-search.ipynb) and [BST](../trees/binary-search-tree.ipynb) notebooks.

For true `O(log n)` ordered operations, the de-facto choice is the third-party **`sortedcontainers.SortedList`** (pure Python, but uses a list-of-lists with large fan-out that beats a textbook AVL in practice due to cache locality):

| Operation | `SortedList` |
|-----------|--------------|
| `add(x)` | `O(log n)` amortized |
| `remove(x)` | `O(log n)` amortized |
| `sl[i]` (index) | `O(log n)` |
| `bisect_left/right` | `O(log n)` |

**Why implement AVL by hand then?** To understand *how* a self-balancing tree maintains its invariant -- rotations and balance factors are the foundation for red-black trees (used inside many language runtimes' ordered maps), B-trees (databases and filesystems), and interval trees.